# Fisher-budget continual learning

I wanted to see if a simple Fisher-based protection rule could reduce forgetting during sequential learning.

The setup is D1 → D2 → D3 on CIFAR-10. The labels stay the same, only the visual distribution changes. I compare the method against normal gradient descent and mainly look at how much D2 performance is lost after training on D3.


## Method

The main idea is simple: use Fisher information as a rough estimate of which parameters matter more, then update those parameters less.

For each minibatch I first compute the normal cross-entropy gradient. I also estimate Fisher importance using squared **per-example** gradients:

\[
F_i = \frac{1}{B}\sum_b
\left(\frac{\partial \ell_b}{\partial \theta_i}\right)^2
\]

I normalize Fisher values inside each parameter tensor:

\[
q_i =
\log\left(
1+\frac{F_i}{\operatorname{mean}(F)}
\right)
\]

Each parameter has a protection score \(s_i\). All scores start at 1.

The scores are cumulative, but I keep the total amount of protection fixed:

\[
\sum_i s_i = N
\]

where \(N\) is the number of trainable parameters.

New Fisher evidence adds a claim

\[
a_i =
q_i
\frac{\exp(-s_i/\tau)}{\tau}
\]

and I update the scores with

\[
\tilde{s}_i = s_i + \beta a_i
\]

After that, the score vector is projected back so that all scores are nonnegative and still sum to \(N\).

So protection can move around between parameters, but it cannot just keep growing forever.

The score is turned into a gradient gate:

\[
r_i =
\frac{1}{
1+\lambda(1-\exp(-s_i/\tau))
}
\]

A higher score means a smaller update.

One problem is that this would normally make the total gradient smaller. To avoid that, I rescale the gated gradient so it has the same global L2 norm as the original gradient:

\[
c =
\frac{\|g\|_2}{\|r\odot g\|_2}
\]

The final update is

\[
\theta_i
\leftarrow
\theta_i-\eta c r_i g_i
\]

So the method changes **where** the update goes, rather than just making the whole step smaller.

The Fisher scores are updated after each weight update, so the current minibatch affects protection on later minibatches.

### Sequential setup

The model is trained on three visual versions of CIFAR-10:

- **D1:** normal CIFAR-10
- **D2:** grayscale with a brightness/contrast change
- **D3:** color tint plus a small amount of Gaussian noise

The classes do not change.

Both methods start from the same initialization and see the same sample order. Each domain uses 8,192 training images, with no source-image overlap between D1, D2, and D3.

Main settings:

- batch size: 128
- learning rate: 0.035
- \(\lambda = 9\)
- \(\beta = 0.2\)
- \(\tau = 2\)
- maximum 500 epochs per stage
- early stopping patience: 10

The model restores the checkpoint with the lowest validation loss after each stage.

The learning rule itself does not get a task or domain label. The experiment still has explicit D1, D2, and D3 stages for validation and stopping.


In [ ]:
# Settings
SEED = 0
SAMPLES_PER_DOMAIN = 8192

BATCH_SIZE = 128
FISHER_CHUNK_SIZE = 64
LEARNING_RATE = 0.035

PROTECTION_STRENGTH = 9.0
SCORE_TRANSFER = 0.2
SATURATION_SCALE = 2.0

MAX_EPOCHS = 500
EARLY_STOP_PATIENCE = 10
MIN_VAL_LOSS_IMPROVEMENT = 0.001

DATA_DIR = "/kaggle/input/datasets/chargedpeanut/cifar-10"
DEVICE = "cuda"  # this experiment is intended to run on a Kaggle GPU


In [ ]:
from __future__ import annotations

import copy
import pickle
import random
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import functional_call, grad, vmap


class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 24, 3, padding=1),
            nn.GroupNorm(4, 24),
            nn.ReLU(),
            nn.Conv2d(24, 48, 3, stride=2, padding=1),
            nn.GroupNorm(8, 48),
            nn.ReLU(),
            nn.Conv2d(48, 64, 3, stride=2, padding=1),
            nn.GroupNorm(8, 64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)


def load_cifar10(root, seed=0):
    """Load CIFAR-10 from extracted Python batches or the original archive."""
    root = Path(root)
    required = [f"data_batch_{i}" for i in range(1, 6)] + ["test_batch"]
    batches = None

    for folder in {p.parent for p in root.rglob("data_batch_1")}:
        if all((folder / name).is_file() for name in required):
            batches = {
                name: pickle.load((folder / name).open("rb"), encoding="bytes")
                for name in required
            }
            break

    if batches is None:
        archives = [
            p for p in root.rglob("*")
            if p.is_file() and p.name.lower().endswith((".tar", ".tar.gz", ".tgz"))
        ]
        for archive_path in archives:
            try:
                with tarfile.open(archive_path, "r:*") as archive:
                    members = {
                        m.name.rsplit("/", 1)[-1]: m
                        for m in archive.getmembers()
                        if m.isfile() and m.name.rsplit("/", 1)[-1] in required
                    }
                    if not all(name in members for name in required):
                        continue
                    batches = {}
                    for name in required:
                        with archive.extractfile(members[name]) as stream:
                            batches[name] = pickle.load(stream, encoding="bytes")
                    break
            except tarfile.TarError:
                pass

    if batches is None:
        raise FileNotFoundError(f"Could not find CIFAR-10 Python batches under {root}")

    def field(batch, name):
        key = name.encode()
        return batch[key] if key in batch else batch[name]

    train_batches = [batches[f"data_batch_{i}"] for i in range(1, 6)]
    train_x = np.concatenate([field(b, "data") for b in train_batches])
    train_y = np.concatenate([np.asarray(field(b, "labels")) for b in train_batches])
    test_x = np.asarray(field(batches["test_batch"], "data"))
    test_y = np.asarray(field(batches["test_batch"], "labels"))

    train_x = torch.from_numpy(train_x.copy().reshape(-1, 3, 32, 32))
    train_y = torch.from_numpy(train_y.astype(np.int64))
    test_x = torch.from_numpy(test_x.copy().reshape(-1, 3, 32, 32))
    test_y = torch.from_numpy(test_y.astype(np.int64))

    rng = np.random.default_rng(40000 + seed)
    val_indices = np.concatenate([
        rng.choice(np.flatnonzero(train_y.numpy() == k), size=500, replace=False)
        for k in range(10)
    ])
    val_mask = np.zeros(len(train_y), dtype=bool)
    val_mask[val_indices] = True

    train_idx = torch.from_numpy(np.flatnonzero(~val_mask))
    val_idx = torch.from_numpy(np.flatnonzero(val_mask))
    return (
        (train_x[train_idx], train_y[train_idx]),
        (train_x[val_idx], train_y[val_idx]),
        (test_x, test_y),
    )


def transform(images, domain, seed=0, chunk_size=1024):
    """Deterministic label-preserving visual shifts."""
    if domain == "clean":
        return images

    output = []
    for start in range(0, len(images), chunk_size):
        x = images[start:start + chunk_size].float() / 255.0

        if domain == "gray":
            gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
            y = (0.06 + 0.94 * gray).clamp(0, 1).expand(-1, 3, -1, -1)

        elif domain == "tint":
            center = x.mean(dim=(2, 3), keepdim=True)
            gain = torch.tensor([1.12, 0.91, 0.80]).view(1, 3, 1, 1)
            tint = torch.tensor([0.035, 0.010, -0.025]).view(1, 3, 1, 1)
            generator = torch.Generator().manual_seed(seed + start)
            noise = torch.randn(x.shape, generator=generator) * 0.025
            y = (center + (x - center) * gain + tint + noise).clamp(0, 1)

        else:
            raise ValueError(f"Unknown domain: {domain}")

        output.append((255 * y).round().to(torch.uint8))

    return torch.cat(output)


def make_schedules(labels, samples_per_domain, seed):
    """Balanced, non-overlapping source images for D1, D2, and D3."""
    y = labels.numpy()
    classes = np.unique(y)
    rng = np.random.default_rng(seed)

    buckets = {k: rng.permutation(np.flatnonzero(y == k)) for k in classes}
    cursor = {k: 0 for k in classes}
    schedules = {}

    for phase in ("d1", "d2", "d3"):
        base, remainder = divmod(samples_per_domain, len(classes))
        quotas = np.full(len(classes), base, dtype=int)
        quotas[:remainder] += 1

        parts = []
        for klass, quota in zip(classes, quotas):
            start = cursor[klass]
            stop = start + quota
            parts.append(buckets[klass][start:stop])
            cursor[klass] = stop

        indices = np.concatenate(parts)
        rng.shuffle(indices)
        schedules[phase] = indices

    combined = np.concatenate(list(schedules.values()))
    if len(np.unique(combined)) != len(combined):
        raise RuntimeError("D1, D2, and D3 schedules overlap")

    return schedules


@torch.inference_mode()
def evaluate(model, data, batch_size=256):
    images, labels = data
    correct = torch.zeros(10, dtype=torch.long)
    counts = torch.zeros(10, dtype=torch.long)
    losses = torch.zeros(10, dtype=torch.float64)

    model.eval()
    for start in range(0, len(labels), batch_size):
        x = images[start:start + batch_size].to(DEVICE).float() / 255.0
        y = labels[start:start + batch_size].to(DEVICE)

        logits = model(x)
        per_example_loss = F.cross_entropy(logits, y, reduction="none").cpu()
        pred = logits.argmax(1).cpu()
        y_cpu = y.cpu()

        for k in range(10):
            mask = y_cpu == k
            counts[k] += mask.sum()
            correct[k] += (pred[mask] == k).sum()
            losses[k] += per_example_loss[mask].double().sum()

    return {
        "accuracy": float((correct / counts).mean()),
        "loss": float((losses / counts).mean()),
    }


def empirical_fisher(model, x, y):
    """Mean squared per-example cross-entropy gradients."""
    params = dict(model.named_parameters())
    buffers = dict(model.named_buffers())

    def one_loss(p, b, one_x, one_y):
        logits = functional_call(model, (p, b), (one_x.unsqueeze(0),))
        return F.cross_entropy(logits, one_y.unsqueeze(0))

    per_example_grads = vmap(
        grad(one_loss),
        in_dims=(None, None, 0, 0),
        chunk_size=FISHER_CHUNK_SIZE,
    )(params, buffers, x, y)

    return {
        name: gradients.detach().square().mean(0)
        for name, gradients in per_example_grads.items()
    }


def normalized_fisher(fisher, names):
    """Normalize Fisher evidence independently inside each parameter tensor."""
    return torch.cat([
        torch.log1p(
            fisher[name].flatten() / fisher[name].mean().clamp_min(1e-12)
        )
        for name in names
    ])


def project_to_budget(scores, budget):
    """Euclidean projection onto s >= 0 with sum(s) = budget."""
    ordered = torch.sort(scores, descending=True).values
    cutoffs = (ordered.cumsum(0) - budget) / torch.arange(
        1, scores.numel() + 1, device=scores.device, dtype=scores.dtype
    )
    rho = torch.nonzero(ordered > cutoffs, as_tuple=False)[-1, 0]
    theta = cutoffs[rho]
    return torch.clamp(scores - theta, min=0)


def train_step(model, x, y, scores, method):
    model.train()
    model.zero_grad(set_to_none=True)

    loss = F.cross_entropy(model(x), y)
    loss.backward()

    params = dict(model.named_parameters())
    names = list(params)
    lengths = [params[name].numel() for name in names]

    if method == "fisher":
        fisher = empirical_fisher(model, x, y)
        evidence = normalized_fisher(fisher, names)

        gate = 1.0 / (
            1.0
            + PROTECTION_STRENGTH
            * (1.0 - torch.exp(-scores / SATURATION_SCALE))
        )
    elif method == "sgd":
        evidence = None
        gate = torch.ones_like(scores)
    else:
        raise ValueError(method)

    raw_norm_sq = sum(params[name].grad.square().sum() for name in names)

    gated_norm_sq = torch.zeros((), device=DEVICE)
    offset = 0
    for name, length in zip(names, lengths):
        p = params[name]
        local_gate = gate[offset:offset + length].view_as(p)
        gated_norm_sq += (p.grad * local_gate).square().sum()
        offset += length

    scale = raw_norm_sq.sqrt() / gated_norm_sq.sqrt().clamp_min(1e-12)

    with torch.no_grad():
        offset = 0
        for name, length in zip(names, lengths):
            p = params[name]
            local_gate = gate[offset:offset + length].view_as(p)
            p.add_(p.grad * local_gate, alpha=-LEARNING_RATE * float(scale))
            offset += length

        if method == "fisher":
            claim = (
                evidence
                * torch.exp(-scores / SATURATION_SCALE)
                / SATURATION_SCALE
            )
            scores = project_to_budget(
                scores + SCORE_TRANSFER * claim,
                budget=float(scores.numel()),
            )

    return scores


def fit_stage(model, scores, train_images, train_labels, indices, validation, method, phase):
    """Train until validation loss plateaus, then restore the lowest-loss checkpoint."""
    phase_code = {"D1": 101, "D2": 211, "D3": 307}[phase]
    best_loss = float("inf")
    best_model = None
    best_scores = None
    wait = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        rng = np.random.default_rng(SEED * 1_000_003 + phase_code * 10_007 + epoch)
        epoch_indices = indices[rng.permutation(len(indices))]

        for start in range(0, len(epoch_indices), BATCH_SIZE):
            batch_indices = torch.as_tensor(
                epoch_indices[start:start + BATCH_SIZE], dtype=torch.long
            )
            x = train_images[batch_indices].to(DEVICE).float() / 255.0
            y = train_labels[batch_indices].to(DEVICE)
            scores = train_step(model, x, y, scores, method)

        metric = evaluate(model, validation)
        absolute_improvement = metric["loss"] < best_loss
        meaningful_improvement = metric["loss"] < best_loss - MIN_VAL_LOSS_IMPROVEMENT

        if absolute_improvement:
            best_loss = metric["loss"]
            best_model = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            best_scores = scores.detach().cpu().clone()

        wait = 0 if meaningful_improvement else wait + 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"{method:6s} {phase} epoch {epoch:3d} | "
                f"val acc {metric['accuracy']:.3f} | "
                f"val loss {metric['loss']:.4f}"
            )

        if wait >= EARLY_STOP_PATIENCE:
            break

    model.load_state_dict(best_model)
    scores = best_scores.to(DEVICE)
    return scores


def run_retention_comparison():
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU before running the notebook.")

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    train, validation, test = load_cifar10(DATA_DIR, SEED)
    train_x, train_y = train

    train_domains = {
        "D1": train_x,
        "D2": transform(train_x, "gray", SEED),
        "D3": transform(train_x, "tint", SEED + 300),
    }
    val_domains = {
        "D1": validation,
        "D2": (transform(validation[0], "gray", SEED + 500), validation[1]),
        "D3": (transform(validation[0], "tint", SEED + 500), validation[1]),
    }
    test_d3 = (transform(test[0], "tint", SEED + 700), test[1])

    schedules = make_schedules(train_y, SAMPLES_PER_DOMAIN, SEED + 101)

    template = SmallCNN()
    initial_weights = copy.deepcopy(template.state_dict())
    parameter_count = sum(p.numel() for p in template.parameters())

    rows = []

    for method in ("sgd", "fisher"):
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

        model = SmallCNN().to(DEVICE)
        model.load_state_dict(copy.deepcopy(initial_weights))
        scores = torch.ones(parameter_count, device=DEVICE)

        scores = fit_stage(
            model, scores, train_domains["D1"], train_y, schedules["d1"],
            val_domains["D1"], method, "D1"
        )
        scores = fit_stage(
            model, scores, train_domains["D2"], train_y, schedules["d2"],
            val_domains["D2"], method, "D2"
        )

        d2_before = evaluate(model, val_domains["D2"])["accuracy"]

        scores = fit_stage(
            model, scores, train_domains["D3"], train_y, schedules["d3"],
            val_domains["D3"], method, "D3"
        )

        d2_after = evaluate(model, val_domains["D2"])["accuracy"]
        d3_test = evaluate(model, test_d3)["accuracy"]

        rows.append({
            "method": "Fisher-budget" if method == "fisher" else "SGD",
            "D2 before D3 (%)": 100 * d2_before,
            "D2 after D3 (%)": 100 * d2_after,
            "D2 forgetting (points)": 100 * (d2_before - d2_after),
            "D3 test (%)": 100 * d3_test,
        })

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(rows).set_index("method")


In [ ]:
results = run_retention_comparison()
results.round(2)


## Result

For the seed-0 run:

| Method | D2 before D3 | D2 after D3 | D2 forgetting | D3 test |
|---|---:|---:|---:|---:|
| SGD | 51.66% | 40.90% | 10.76 points | 58.13% |
| Fisher-budget | 53.02% | 48.14% | 4.88 points | 60.34% |

The Fisher-budget model forgot less of D2 after learning D3: a 4.88-point drop versus 10.76 points for SGD.

It also did slightly better on D3, so the lower forgetting was not just because it avoided learning the new distribution.

This is only one seed, so I would treat it as a promising result rather than a strong conclusion.
